<a href="https://colab.research.google.com/github/ksuplee/tensorflow-nlp-tutorial/blob/main/12_HuggingFace/12_01_chat_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Hugging Face 추론 API 사용 (OpenAI 방식과 가장 유사)

Hugging Face의 InferenceClient는 OpenAI의 API 구조와 거의 동일하게 설계되어 있어 코드 변경을 최소화할 수 있습니다. 무거운 모델을 직접 다운로드할 필요가 없습니다.

사전 준비:

Colab 왼쪽 🔑(보안 비밀) 탭에 HF_TOKEN이라는 이름으로 Hugging Face Access Token을 저장해야 합니다.

!pip install huggingface_hub 실행 필요

In [ ]:
!pip install huggingface_hub

In [5]:
from huggingface_hub import InferenceClient
from google.colab import userdata

# Colab 시크릿에서 Hugging Face 토큰 불러오기
HF_TOKEN = userdata.get('HF_TOKEN')

# OpenAI Client와 동일한 구조의 InferenceClient 초기화
# 한국어 성능이 좋은 오픈소스 모델(예: Qwen2.5 또는 Gemma2) 지정
client = InferenceClient(
    model="Qwen/Qwen2.5-72B-Instruct",
    token=HF_TOKEN
)

resp = client.chat.completions.create(
    messages=[
        {"role": "system", "content": "너는 유능한 AI 어시스턴트다."},
        {"role": "user",   "content": "트랜스포머의 핵심 아이디어를 한 문장으로 설명해줘."},
        {"role": "user",   "content": "창발 현상에 대해 설명해줘."},
    ],
    temperature=0.3,
    max_tokens=100,
)

print(resp.choices[0].message.content)

트랜스포머의 핵심 아이디어는 self-attention 메커니즘을 통해 입력 시퀀스의 모든 단어 간의 관계를 평행하게 계산하는 것입니다.

창발 현상(Emergent Phenomenon)은 복잡한 시스템에서 개별 구성 요소들이 상호작용하면서 예상치 못한 새로운 특성이나 행동이 나타나는 현상을 말합니다. 이는 인공지


# 2. Transformers 라이브러리를 통한 로컬 GPU 구동

외부 서버에 의존하지 않고 Colab의 GPU 자원을 직접 사용하여 모델을 돌리고 싶을 때 사용하는 방식입니다.

사전 준비:

런타임 유형을 GPU로 변경해야 합니다.

!pip install transformers accelerate 실행 필요

In [1]:
!pip install transformers accelerate

In [5]:
import torch
from transformers import pipeline
from google.colab import userdata

# 제한된(Gated) 모델을 사용할 경우 토큰 필요
HF_TOKEN = userdata.get('HF_TOKEN')

# 텍스트 생성 파이프라인 구축 (Colab 무료 GPU에 맞는 7B~9B 사이즈 권장)
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", # 접근 가능한 공개 모델로 변경
    token=HF_TOKEN,
    device_map="auto",
    torch_dtype=torch.float16 # 메모리 절약을 위해 16bit 정밀도 사용
)

messages = [
    {"role": "system", "content": "너는 유능한 AI 어시스턴트다."},
    {"role": "user",   "content": "트랜스포머의 핵심 아이디어를 한 문장으로 설명해줘."},
    {"role": "user",   "content": "디코더 기반 자기회귀 생성을 설명해줘."},
]

# 파이프라인에 메시지 전달
result = pipe(
    messages,
    max_new_tokens=100,
    temperature=0.3,
    do_sample=True # temperature를 적용하기 위해 필요
)

# 생성된 답변의 텍스트만 추출
print(result[0]['generated_text'][-1]['content'])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


디코더 기반 자기회귀 생성 (Deep Recurrent Neural Networks, DRNN)는 네트워크 구조를 잘 알고 있는 딥러닝 모델을 사용하는 것입니다.


```markdown
To securely use your OpenAI API key in Colab, you can store it in the Colab secrets manager. Click the '🔑' icon in the left panel, add a new secret, and name it `OPENAI_API_KEY`. Then, you can access it in your code as shown below.
```

# 1. 코드 1 : 챗봇

In [ ]:
from openai import OpenAI
client = OpenAI(api_key="YOUR_KEY")   # 키는 환경변수로 관리 권장

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "너는 유능한 AI 어시스턴트다."},
        {"role": "user",   "content": "트랜스포머의 핵심 아이디어를 한 문장으로 설명해줘."},
    ],
    temperature=0.3,
    max_tokens=100,
)
print(resp.choices[0].message.content)


# 2. 코드 2 : 보안키 처리 코드

In [3]:
from openai import OpenAI
from google.colab import userdata

# Retrieve the API key from Colab secrets
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "너는 유능한 AI 어시스턴트다."}, # You are a capable AI assistant.
        {"role": "user",   "content": "트랜스포머의 핵심 아이디어를 한 문장으로 설명해줘."}, # Explain the core idea of Transformers in one sentence.
    ],
    temperature=0.3,
    max_tokens=100,
)
print(resp.choices[0].message.content)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}